# Generate candidate scenes (Colab, free T4 GPU)

The writer model continues each of the 12 scene openings in `prompts.jsonl`, **256 times each**
(3,072 scenes). Back on the laptop, the gold and proxy critics score every candidate, and
best-of-N is computed exactly from these pools, for any N up to 256.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.
You will be asked to upload `data/processed/prompts.jsonl`. At the end, `candidates.jsonl`
and `generation_meta.json` download automatically; put both in `data/generated/`.

Expected time: ~5 min install, ~15–25 min generation.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q vllm

In [ ]:
import json, time
from google.colab import files

uploaded = files.upload()                      # choose data/processed/prompts.jsonl
prompts = [json.loads(l) for l in open("prompts.jsonl", encoding="utf-8")]
print(len(prompts), "prompts")

In [ ]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # small, Apache-2.0, fits a T4 easily
M = 256                                # candidates per prompt (best-of-N up to 256)
TEMPERATURE = 1.0                      # plain sampling: diverse candidates
TOP_P = 0.95
MAX_TOKENS = 800                       # ~450 words of dialogue is ~700 tokens
SEED = 42

SYSTEM = "You are a staff writer on {show}, {genre}."
USER = (
    "Continue the scene below. Write the next ~450 words in the same transcript format: "
    "each spoken line as 'NAME: line', stage directions in [brackets]. Keep the same "
    "characters and tone. Output only the continuation, with no title, notes or preamble."
    "\n\nSCENE SO FAR:\n{opening}"
)

In [ ]:
from vllm import LLM, SamplingParams
import vllm

llm = LLM(model=MODEL, dtype="half", max_model_len=2048, gpu_memory_utilization=0.90, seed=SEED)
tok = llm.get_tokenizer()

texts, params = [], []
for p in prompts:
    msgs = [{"role": "system", "content": SYSTEM.format(**p)},
            {"role": "user", "content": USER.format(**p)}]
    texts.append(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
    params.append(SamplingParams(n=M, temperature=TEMPERATURE, top_p=TOP_P,
                                 max_tokens=MAX_TOKENS, seed=SEED + p["prompt_id"]))
print(texts[0][:600])

In [ ]:
t0 = time.time()
results = llm.generate(texts, params)
elapsed = time.time() - t0

n_tok = 0
with open("candidates.jsonl", "w", encoding="utf-8") as f:
    for p, r in zip(prompts, results):
        for i, o in enumerate(r.outputs):
            n_tok += len(o.token_ids)
            f.write(json.dumps({"prompt_id": p["prompt_id"], "cand_id": i, "text": o.text,
                                "finish_reason": o.finish_reason,
                                "n_tokens": len(o.token_ids)}, ensure_ascii=False) + "\n")

meta = dict(model=MODEL, vllm_version=vllm.__version__, M=M, temperature=TEMPERATURE,
            top_p=TOP_P, max_tokens=MAX_TOKENS, seed=SEED, system=SYSTEM, user=USER,
            n_prompts=len(prompts), n_candidates=len(prompts) * M, total_tokens=n_tok,
            seconds=round(elapsed, 1))
json.dump(meta, open("generation_meta.json", "w"), indent=2)
print(f"{len(prompts) * M} candidates, {n_tok:,} tokens in {elapsed / 60:.1f} min "
      f"({n_tok / elapsed:,.0f} tokens/s)")

In [ ]:
import collections, statistics
rows = [json.loads(l) for l in open("candidates.jsonl", encoding="utf-8")]
words = [len(r["text"].split()) for r in rows]
print("words per candidate: median", statistics.median(words), "| min", min(words), "| max", max(words))
print("finish reasons:", collections.Counter(r["finish_reason"] for r in rows))
for r in rows[:2]:
    print("\n----- prompt", r["prompt_id"], "cand", r["cand_id"], "-----\n", r["text"][:800])

In [ ]:
files.download("candidates.jsonl")
files.download("generation_meta.json")